# FRE-GY-9743 Assignment 5

## Part I: Implicit Function Differentiation

Suppose you have a parametric model $M(X^I)$, where $X^I$ is a vector.
And you have a vector of functions $V$, where each atomic function $V_i(X^I) = X^M_j,\ \forall j$, and $\dim(X^I) = \dim(X^M)$.

### Q1.

What's your general recipe for solving $X^I$ from $V(X^I) = X^M$?

**Answer (Q1).** Treat it as a square root-finding (calibration) problem: $n$ equations, $n$ unknowns.

1. **Residual.** $R(X^I) = V(X^I) - X^M$, and we want $R(X^I) = 0$.
2. **Initial guess.** Start close to the answer: yesterday's calibrated $X^I$, a flat curve / flat vol, or a quick bootstrap.
3. **Newton–Raphson.** With Jacobian $J(X^I) = \dfrac{\partial V}{\partial X^I}$ ($n \times n$),
$$J(X^I_k)\,\Delta_k = -R(X^I_k), \qquad X^I_{k+1} = X^I_k + \lambda_k \Delta_k .$$
   Solve the linear system (LU) instead of inverting $J$. Use an analytic Jacobian (chain rule / AAD) if possible; otherwise finite differences or Broyden updates.
4. **Robustness.** Line search / damping $\lambda_k \in (0, 1]$ so that $\|R\|$ decreases; Levenberg–Marquardt when $J$ is ill-conditioned; bounds or transforms for constrained parameters (e.g. $\sigma > 0$, $|\rho| < 1$).
5. **Stopping rule.** $\|R\|_\infty < \varepsilon$ (in quote units, e.g. $10^{-10}$) and a small step; cap the number of iterations and report failure explicitly.
6. **Exploit structure.** If $V_i$ only depends on $X^I_1, \dots, X^I_i$ (node-by-node curve bootstrapping, lower-triangular $J$), solve sequentially with 1-D root finders (Newton / Brent). If the interpolation couples the nodes, solve all of them jointly with the global Newton above.
7. **Equivalent view.** Minimise $\tfrac12 \|V(X^I) - X^M\|^2$ (Gauss–Newton / LM). For a well-posed square system the minimum is $0$ and coincides with the root; this formulation also covers the over-determined (more quotes than parameters) case.

Well-posedness requires $J$ to be non-singular around the solution, i.e. every quote pins down a different direction of $X^I$. This is the same condition the implicit function theorem needs in Q2.

### Q2.

How do you calculate $\dfrac{\mathrm{d}X^I}{\mathrm{d}X^M}$ at optimal $X^I$?

**Answer (Q2).** Use the implicit function theorem; no re-calibration is needed.

At the solution $X^{I*}$ the calibration identity holds for all market states in a neighbourhood of $X^M$:
$$V\big(X^I(X^M)\big) - X^M \equiv 0 .$$
Differentiate with respect to $X^M$:
$$\frac{\partial V}{\partial X^I}\bigg|_{X^{I*}} \frac{\mathrm{d}X^I}{\mathrm{d}X^M} - \mathbb{I} = 0
\quad\Longrightarrow\quad
\boxed{\frac{\mathrm{d}X^I}{\mathrm{d}X^M} = \left(\frac{\partial V}{\partial X^I}\bigg|_{X^{I*}}\right)^{-1} = J^{-1}} .$$
More generally, for $F(X^I, X^M) = 0$: $\dfrac{\mathrm{d}X^I}{\mathrm{d}X^M} = -\left(\dfrac{\partial F}{\partial X^I}\right)^{-1}\dfrac{\partial F}{\partial X^M}$. This requires $V \in C^1$ and $J$ invertible at $X^{I*}$.

Practical notes:

- $J$ at the solution is already available from the last Newton step. One LU factorisation gives all $n$ columns. The result is exact up to the accuracy of $J$, whereas bump-and-recalibrate costs $n$ extra calibrations and picks up solver-tolerance noise.
- **Risk to market quotes.** For a product, $\dfrac{\mathrm{d}PV}{\mathrm{d}X^M} = \dfrac{\partial PV}{\partial X^I}\,J^{-1}$. Compute it adjoint-style: solve $J^{\top}\lambda = \left(\dfrac{\partial PV}{\partial X^I}\right)^{\top}$, one linear solve per product whatever $n$ is, and $\lambda$ is the market-quote risk. This is the pipeline of this library: the valuation engine produces $\partial PV / \partial(\text{state})$, and the model's Jacobian converts state risk into quote risk. In this HW the states *are* the quoted IFRs, so $J = \mathbb{I}$ and the conversion is the identity.
- **Least-squares calibration** (non-zero residual at the optimum): the optimality condition is $G = J^{\top}\big(V(X^I) - X^M\big) = 0$. Applying the IFT to $G$ gives
  $\dfrac{\mathrm{d}X^I}{\mathrm{d}X^M} = \Big(J^{\top}J + \sum_i r_i \nabla^2 V_i\Big)^{-1} J^{\top} \approx (J^{\top}J)^{-1}J^{\top}$, which reduces to $J^{-1}$ for a square system with an exact fit.

### Q3.

Describe how you apply the above to the following problem:

- You observe 3 European OTM option un-discounted prices in the market $P_1, P_2, P_3$, which correspond to $K_1 < K_{\text{atm}} < K_3$ ($P_2 \Rightarrow K_{\text{atm}}$).
- You have an interpolator in the implied vol space $\sigma(K;\theta)$ that returns you an implied vol per strike. $\theta$ is of course the parameters of the vol interpolator.
- You want to find the unique $\theta$ that is consistent with your market observation, how? You want to find how $\sigma(K)$ changes due to change of $\theta$?
- To make the sensitivity of $\sigma(K)$ w.r.t. $\theta$ more intuitive to trader, what do you think $\theta$ might correspond to? (just intuitively)

*No computation is needed, it's an open question, just write down what you think.*

**Answer (Q3).**

**Mapping to Q1/Q2.** $X^M = (P_1, P_2, P_3)$ and $X^I = \theta \in \mathbb{R}^3$ (a 3-parameter interpolator, so the system is square). The model functions are
$$V_i(\theta) = \text{Black}\big(F, K_i, T, \sigma(K_i;\theta)\big) \quad (\text{un-discounted}),$$
a put at $K_1$, a call at $K_3$, and either at $K_{\text{atm}}$ (they coincide at the ATM forward by put–call parity).

**1. Find $\theta$ (calibration).**

- First convert prices to implied vols, $\sigma_i^{\text{mkt}} = \text{Black}^{-1}(P_i; K_i)$. Each is a 1-D root find, and it is well-defined because Black is monotone in $\sigma$ (vega $> 0$). Then solve
  $$\sigma(K_i;\theta) = \sigma_i^{\text{mkt}}, \quad i = 1, 2, 3 .$$
  The solution is the same (the map is a bijection), but vol space is better scaled across strikes and usually close to linear in $\theta$.
- Run Newton on $\theta$ with $J_{ij} = \partial \sigma(K_i;\theta) / \partial \theta_j$. In price space this is $\partial V_i / \partial \theta_j = \text{Vega}_i \, \partial \sigma(K_i;\theta) / \partial \theta_j$. Start from a flat smile at the ATM vol.
- Uniqueness needs $J$ non-singular: distinct strikes, and each parameter moves the smile in a different way (level, tilt, bend). After the fit, check for no arbitrage (call prices convex in $K$, non-negative density).

**2. How $\sigma(K)$ moves.**

- $\partial \sigma(K) / \partial \theta = \nabla_\theta \sigma(K;\theta)$ is the analytic gradient of the interpolator at any strike $K$.
- From Q2: $\dfrac{\mathrm{d}\theta}{\mathrm{d}\sigma^{\text{mkt}}} = \big[\partial_\theta \sigma(K_i;\theta)\big]^{-1}$, or in price space $\dfrac{\mathrm{d}\theta}{\mathrm{d}P} = \big[\operatorname{diag}(\text{Vega}_i)\,\partial_\theta \sigma(K_i;\theta)\big]^{-1}$.
- Combined:
  $$w_j(K) := \frac{\mathrm{d}\sigma(K)}{\mathrm{d}\sigma_j^{\text{mkt}}} = \nabla_\theta \sigma(K;\theta)\,\big[\partial_\theta \sigma(K_i;\theta)\big]^{-1}_{\cdot j},$$
  which is the weight of market quote $j$ in the vol at strike $K$. Sanity check: $w_j(K_i) = \delta_{ij}$. An option struck at $K$ with Black vega $\nu(K)$ has bucketed vega $\nu(K)\,w_j(K)$ to quote $j$, i.e. it is hedged with $\nu(K)\,w_j(K) / \text{Vega}_j$ units of market option $j$.

**3. What $\theta$ should mean to a trader.** Traders read a smile through three handles, **level, skew and curvature** (the analogue of level / slope / curvature of a yield curve). So $\theta$ should be, or be re-parametrised into:

| $\theta$ | Trader meaning | Shape of $\partial\sigma(K)/\partial\theta$ |
|---|---|---|
| $\theta_1$ | ATM vol $\sigma_{\text{atm}}$ (level) | $\approx 1$ for all $K$: parallel shift |
| $\theta_2$ | Skew / risk reversal $RR = \sigma(K_3) - \sigma(K_1)$ | antisymmetric in log-moneyness: tilt |
| $\theta_3$ | Curvature / butterfly $BF = \tfrac12\big(\sigma(K_1) + \sigma(K_3)\big) - \sigma_{\text{atm}}$ | symmetric, zero at ATM: bend |

This is how FX vol is quoted (ATM, 25Δ RR, 25Δ BF). For example, take $\sigma(K;\theta) = a + b\,x + c\,x^2$ with $x = \ln(K/F)$ and $x_{1,3} = \mp h$. Then $a = \sigma_{\text{atm}}$, $b = RR / (2h)$ and $c = BF / h^2$, so $\theta$ is literally (ATM, RR, BF) up to scaling. Here $J$ is a Vandermonde matrix, non-singular for distinct strikes, so the fit is unique (and a single linear solve). For SABR the same reading is $\alpha \leftrightarrow$ level, $\rho \leftrightarrow$ skew and $\nu$ (vol-of-vol) $\leftrightarrow$ curvature, with $\beta$ fixed.

Another trader-friendly choice is to take $\theta = (\sigma_1, \sigma_{\text{atm}}, \sigma_3)$, the pillar vols themselves. Then $J = \mathbb{I}$, and the sensitivity to $\theta$ is directly the vega bucketed to the three quoted options, which are exactly the hedging instruments.

## Part II: Implement a Valuation Engine

Github Link: https://github.com/Jerry0108-SJ/FRE-GY-9743-Assignments-5.git

Please follow the instruction in the notebook and complete the code.

# HW5: Fixed-Income Valuation and First-Order Risk

Complete the five library TODOs, then run all cells.

In [1]:
from typing import cast

import numpy as np
import pandas as pd
from IPython.display import display

from fixedincomelib import (
    qfCreateData1D,
    qfCreateDataCollection,
    qfCreateBuildMethod,
    qfCreateModelBuildMethodCollection,
    qfCreateModel,
    qfCreateProductFixedAccruedCashflow,
    qfCreateValuationParameters,
    qfCreateValuationParametersCollection,
    qfCreateValueReport,
)
from fixedincomelib.date import Date
from fixedincomelib.market.registries import IndexRegistry
from fixedincomelib.valuation import ValuationRequest
from fixedincomelib.yield_curve import YieldCurveModelComponent
from fixedincomelib.yield_curve.valuation_engine import ValuationEngineProductFixedAccrued

np.set_printoptions(precision=6, suppress=True)
pd.options.display.float_format = "{:,.6f}".format
pd.options.display.max_colwidth = None

## 1. Input data

Rates are annual decimals: `0.03 = 3%`. Edit the inputs here and rerun the following cells.
The funding component adds an IFR spread over SOFR.

In [2]:
VALUE_DATE = "2026-01-02"
PAYMENT_DATE = Date("2026-10-02")
FUNDING_INDEX = "SOFR-1B-FLAT"

sofr_quotes = pd.DataFrame(
    {"values": [0.03, 0.04, 0.05]},
    index=pd.Index(["2026-04-02", "2027-01-02", "2028-01-02"], name="Node date"),
)
funding_quotes = pd.DataFrame(
    {"values": [0.0, 0.001]},
    index=pd.Index(["2026-07-02", "2027-01-02"], name="Node date"),
)

display(sofr_quotes.rename(columns={"values": "SOFR-1B IFR"}))
display(funding_quotes.rename(columns={"values": "SOFR-1B-FLAT IFR spread"}))

,SOFR-1B IFR
Node date,
2026-04-02,0.030000
2027-01-02,0.040000
2028-01-02,0.050000


,SOFR-1B-FLAT IFR spread
Node date,
2026-07-02,0.000000
2027-01-02,0.001000


## 2. Data and build methods

Create the IFR datasets and build methods, with SOFR first and funding second.

In [3]:
sofr_data = qfCreateData1D("INSTANTANEOUS FORWARD RATE", "SOFR-1B-IFR", sofr_quotes)
funding_data = qfCreateData1D(
    "INSTANTANEOUS FORWARD RATE", "SOFR-1B-FUNDING-IFR", funding_quotes,
)
data_collection = qfCreateDataCollection([sofr_data, funding_data])
display(data_collection.display())

,Data Shape,Data Type,Data Convention
0,DATA1D,INSTANTANEOUS FORWARD RATE,SOFR-1B-IFR
1,DATA1D,INSTANTANEOUS FORWARD RATE,SOFR-1B-FUNDING-IFR


In [4]:
sofr_method = qfCreateBuildMethod("YIELD_CURVE_INDEX", {
    "TARGET": "SOFR-1B", "INSTANTANEOUS FORWARD RATE": "SOFR-1B-IFR",
})
funding_method = qfCreateBuildMethod("YIELD_CURVE_FUNDING", {
    "TARGET": FUNDING_INDEX, "REFERENCE INDEX": "SOFR-1B",
    "INSTANTANEOUS FORWARD RATE": "SOFR-1B-FUNDING-IFR",
})
build_methods = qfCreateModelBuildMethodCollection([sofr_method, funding_method])

## 3. Build the model

Copy IFR inputs into the model. The registered funding component references SOFR.

In [5]:
model = qfCreateModel(VALUE_DATE, "YIELD_CURVE", data_collection, build_methods)
print("Component sizes:", model.num_sub_components)

Component sizes: [3, 2]


The following helpers retrieve a component (`retrieve_model_component`), allocate zero-filled
gradient blocks in component order (`resize_gradient`), and locate a component's block
(`retrieve_model_component_index`, using the `component_indices` map).

In [6]:
index = IndexRegistry().get("SOFR-1B")
component = cast(YieldCurveModelComponent, model.retrieve_model_component(index))
funding_component = model.retrieve_model_component(FUNDING_INDEX)

print("SOFR-1B ->", type(component).__name__, "|", component.component_identifier.name())
print("SOFR-1B-FLAT ->", funding_component.component_identifier.name())
display(pd.DataFrame(component.state_data.T, columns=["Node time (years)", "IFR"]))

SOFR-1B -> YieldCurveModelComponent | SOFRON Actual/360
SOFR-1B-FLAT -> SOFR-1B-FLAT


,Node time (years),IFR
0,0.246575,0.030000
1,1.000000,0.040000
2,1.999993,0.050000


In [7]:
gradient = []
model.resize_gradient(gradient)
print("[SOFR block, funding block]:", gradient)

[SOFR block, funding block]: [array([0., 0., 0.]), array([0., 0.])]


In [8]:
print("Stored map:", model.component_indices)
print("SOFR-1B -> block", model.retrieve_model_component_index("SOFR-1B"))
print("SOFR-1B-FLAT -> block", model.retrieve_model_component_index(FUNDING_INDEX))

Stored map: {'SOFRON Actual/360': 0, 'SOFR-1B-FLAT': 1}
SOFR-1B -> block 0
SOFR-1B-FLAT -> block 1


## 4. Discount-factor gradients

Complete **TODO 1** in `YieldCurveModelComponent` and **TODO 2** in `YieldCurve`
(`fixedincomelib/yield_curve/yield_curve_model.py`). Calculate the local derivative, then apply
the product rule recursively to $D_{\mathrm{funding}}=D_{\mathrm{spread}}D_{\mathrm{SOFR}}$.

Gradient methods write raw derivatives into the supplied container and return `None`.
`scaler` scales the new contribution; `accumulate=False` replaces the output and `True` adds to it.

With the default piecewise-constant interpolation, the third SOFR segment starts after
2027-01-02. Its gradient is zero for the 2026-10-02 payment.

In [9]:
local_gradient = np.zeros(component.num_state_data)
component.discount_factor_gradient_wrt_state(PAYMENT_DATE, local_gradient)
print("Local SOFR discount-factor gradient:", local_gradient)

Local SOFR discount-factor gradient: [-0.239898 -0.487793 -0.      ]


In [10]:
model.discount_factor_gradient_wrt_state(FUNDING_INDEX, PAYMENT_DATE, gradient)
print("Model gradient [SOFR block, funding block]:", gradient)

Model gradient [SOFR block, funding block]: [array([-0.239838, -0.48767 ,  0.      ]), array([-0.482341, -0.245168])]


## 5. Fixed-accrued valuation engine

Complete **TODOs 3–5** in `ValuationEngineProductFixedAccrued`: `__init__`, `calculate_value`,
and `calculate_risk` (`fixedincomelib/yield_curve/valuation_engine.py`).
Select the funding curve, then compute $C=N\alpha$, $PV=C D(T_p)$ and
$\nabla PV=C\nabla D(T_p)$. Notional is signed; this product has no separate fixed-rate field.

| Payment date | PV / cash today | Risk |
|---|---|---|
| Future | $C D(T_p)$ / 0 | $C\nabla D(T_p)$ |
| Today | $C$ / $C$ | 0 |
| Past | 0 / 0 | 0 |

In [11]:
product = qfCreateProductFixedAccruedCashflow(
    "2026-04-02", "2026-07-02", "USD", 1_000_000, "ACT/360",
    payment_date=PAYMENT_DATE.ISO(), business_day_convention="NONE",
)
funding_parameter = qfCreateValuationParameters(
    "FUNDING INDEX PARAMETER", {"FUNDING INDEX": FUNDING_INDEX},
)
parameters = qfCreateValuationParametersCollection([funding_parameter])

print("Accrual fraction:", product.accrued)
print("Cash amount:", product.notional * product.accrued)

Accrual fraction: 0.25277777777777777
Cash amount: 252777.77777777778


In [12]:
engine = ValuationEngineProductFixedAccrued(model, parameters, product, ValuationRequest.PV)
engine.calculate_value()

product_gradient = []
engine.calculate_risk(product_gradient)

print("PV:", engine.value)
print("Cash today:", engine.cash)
print("Product gradient [SOFR block, funding block]:", product_gradient)

PV: 245870.91057967892
Cash today: 0.0
Product gradient [SOFR block, funding block]: [array([ -60625.703979, -123272.264756,       0.      ]), array([-121925.02689 ,  -61972.941845])]


## 6. Reports

`qfCreateValueReport` calls the engine for `PV` or `FirstOrderRisk`.
`state_gradient` retains raw blocks; `display()` multiplies risk by `0.0001`
to show the signed PV change per positive 1 bp node shift.

In [13]:
print("PV report:", qfCreateValueReport(model, product, parameters, "PV"))
risk_report = qfCreateValueReport(model, product, parameters, "FirstOrderRisk")

print("Raw blocks:", risk_report.state_gradient)
display(risk_report.display()[["COMPONENT", "NODE_INDEX", "AXIS1", "UNIT", "VALUES"]])

PV report: [['USD', 245870.91057967892]]
Raw blocks: [array([ -60625.703979, -123272.264756,       0.      ]), array([-121925.02689 ,  -61972.941845])]


,COMPONENT,NODE_INDEX,AXIS1,UNIT,VALUES
0,SOFRON Actual/360,0,2026-04-02,0.000100,-6.062570
1,SOFRON Actual/360,1,2027-01-02,0.000100,-12.327226
2,SOFRON Actual/360,2,2028-01-02,0.000100,0.000000
3,SOFR-1B-FLAT,0,2026-07-02,0.000100,-12.192503
4,SOFR-1B-FLAT,1,2027-01-02,0.000100,-6.197294
